# Local CPU Fine-Tuning: Llama 3.1 8B (PEFT/LoRA)

**⚠️ CRITICAL WARNING: CPU TRAINING IS EXTREMELY SLOW ⚠️**
Training an 8B parameter model like Llama 3.1 on a standard CPU will take weeks or even months for 10,000 records. QLoRA (`bitsandbytes` 4-bit) relies heavily on NVIDIA CUDA GPUs for training. 

This notebook provides the setup to train using standard LoRA with Hugging Face `transformers` on CPU, optimized as much as possible for memory footprint (using gradient checkpointing and small batch sizes). If `bitsandbytes` fails to load 4-bit on CPU, it falls back to standard precision.

If you want practical training times, please consider using Google Colab (with a smaller dataset or checkpoints), RunPod, or Lambda Labs for an affordable GPU.

In [1]:
!pip install --upgrade "transformers>=4.43.0" peft accelerate datasets trl

In [2]:
import torch
import os
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
dataset_path = "ft_data/erp_whw_10000_balanced.jsonl" # Using your new dataset

# To minimize hardware usage on CPU, we use a small max_seq_length
max_seq_length = 512

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model on CPU (This will take a lot of RAM)...")
# Note: 4-bit bitsandbytes training on CPU is not fully supported.
# We load in standard precision but use memory reduction techniques.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cpu", 
    torch_dtype=torch.float32, # float32 is most compatible for CPU fallback
    low_cpu_mem_usage=True
)

d:\Research\Implementation\Fine-tune-Llama\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer...


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct.
401 Client Error. (Request ID: Root=1-69ff3747-068cad4803c9203275178d36;1b1c5196-86ed-4b06-a1f1-f0fdb6c2773e)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
print("Applying PEFT/LoRA to reduce trainable parameters...")

# Enable gradient checkpointing to save memory at the cost of slower compute
model.gradient_checkpointing_enable()

peft_config = LoraConfig(
    r=8, # Low rank to save memory and compute
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
print("Loading and formatting dataset...")
dataset = load_dataset("json", data_files=dataset_path, split="train")

def format_prompts(examples):
    formatted_texts = []
    for inst, inp, out in zip(examples.get("instruction", []), examples.get("input", []), examples.get("output", [])):
        system_prompt = "You are an expert ERP Finance AI assistant. You process financial events accurately and do not hallucinate outside the provided context."
        
        user_content = inst
        if inp and inp.strip():
            user_content += f"\nContext: {inp}"
            
        # Llama 3 Chat Template formatting
        text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|>"
        text += f"<|start_header_id|>user<|end_header_id|>\n\n{user_content}<|eot_id|>"
        text += f"<|start_header_id|>assistant<|end_header_id|>\n\n{out}<|eot_id|>"
        formatted_texts.append(text)
        
    return { "text" : formatted_texts }

dataset = dataset.map(format_prompts, batched=True)
print("Sample Formatted Text:")
print(dataset[0]['text'])

In [ ]:
print("Setting up Trainer...")

# Split into train/eval for testing accuracy later
dataset_split = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1, # Minimum batch size for CPU
    gradient_accumulation_steps=8, # Accumulate gradients to simulate larger batch
    learning_rate=2e-4,
    logging_steps=1,
    max_steps=100, # Start with a small number of steps to test! 
    # For full training, remove max_steps and set num_train_epochs=1
    evaluation_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    optim="adamw_torch", # Standard PyTorch AdamW for CPU
    report_to="none",
    fp16=False, # CPU does not natively accelerate fp16 well without specific extensions
    bf16=False,
    use_cpu=True
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_args,
)

In [ ]:
# THIS WILL BE EXTREMELY SLOW ON CPU
print("Starting training...")
trainer.train()

In [ ]:
# Save the final LoRA adapters
model.save_pretrained("local_cpu_lora_model")
tokenizer.save_pretrained("local_cpu_lora_model")
print("Training complete and LoRA model saved!")

## Evaluate Model Accuracy
We can evaluate the model by running inference on a few examples from our evaluation set and measuring validation loss.

In [ ]:
import torch

# Evaluate Validation Loss
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

# Test Generation
test_prompt = eval_dataset[0]['text'].split("<|start_header_id|>assistant<|end_header_id|>\n\n")[0] + "<|start_header_id|>assistant<|end_header_id|>\n\n"
inputs = tokenizer(test_prompt, return_tensors="pt").to("cpu")

model.eval()
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
    
print("\n--- Test Generation ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))